# Evaluation of saved predictions

Reproduces the performance, calibration, decision curve, model-updating, logistic regression, and device-statement analyses from saved model predictions. The model definition, training code, and weights are not included.

**Inputs**
- `rebuild_v3/china_v2.npz`, `rebuild_v3/korea_v2.npz` from `data_preparation.ipynb`
- `predictions/oof_predictions.npz`: out-of-fold predictions of the 5 fold models (`rows`, `targets`, `probs`, `fold`)
- `predictions/predictions_internal.npz`, `predictions/predictions_external.npz`: fold-model probabilities for the internal test set and the primary external cohort (`rows`, `y`, `ids`, `fold_probs`, `prob`)
- `predictions/predictions_external_all.npz`: recalibrated fold-model probabilities for all external ECG–potassium pairs within 60 minutes (`rows`, `fold_probs_recal`)

Outputs are written to `results_evaluation/`.

In [ ]:
import os
import copy
import json
import random
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             f1_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from tqdm import tqdm

warnings.filterwarnings("ignore")

SEED = 421
K_FOLDS = 5
TEST_SIZE = 0.15

DATA_DIR = "rebuild_v3"
CHINA_PATH = os.path.join(DATA_DIR, "china_v2.npz")
KOREA_PATH = os.path.join(DATA_DIR, "korea_v2.npz")
PRED_DIR = "predictions"
SAVE_DIR = "results_evaluation"

CROP_MODE = "median"
CROP_KEYS = {
    "median": ("X_crop", "crop_valid_len", "M"),
    "gauss": ("X_crop_gauss", "crop_valid_len", "M_gauss"),
    "legacy": ("X_crop_legacy", "crop_valid_len_legacy", "M_legacy"),
}

TRAIN_EXCLUDE_FLAGS = []
TEST_EXCLUDE_FLAGS = []
EXTERNAL_EXCLUDE_FLAGS = []

EXTERNAL_MAX_DT_MIN = 5
EXTERNAL_AGE_RANGE = (1, 18)
AGE_TXT = f"{EXTERNAL_AGE_RANGE[0]}–{EXTERNAL_AGE_RANGE[1] - 1} y"

TARGET_SENSITIVITY = 0.80
N_BOOTSTRAP = 1000
USE_RECALIBRATED = True

os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def load_cohort(path):
    data = dict(np.load(path, allow_pickle=True))
    crop_key, len_key, meta_key = CROP_KEYS[CROP_MODE]
    if CROP_MODE == "legacy":
        ok = data["legacy_ok"].astype(bool)
        print(f"  {int((~ok).sum())} ECGs without a legacy crop are excluded from the legacy analysis")
    else:
        ok = np.ones(len(data["y"]), dtype=bool)
    data["_crop"] = data[crop_key]
    data["_valid_len"] = data[len_key]
    data["_meta"] = data[meta_key].astype(np.float32)
    data["_ok"] = ok
    return data


def cohort_mask(data, exclude_flags):
    mask = data["_ok"].copy()
    for flag in exclude_flags:
        mask &= ~data[flag].astype(bool)
    return mask


def patient_level_label(ids, y):
    frame = pd.DataFrame({"id": ids, "pos": (np.asarray(y) == 2)})
    grouped = frame.groupby("id")["pos"].max()
    return grouped.index.to_numpy(), grouped.to_numpy().astype(int)


def select_threshold(targets, probabilities, target_sensitivity=TARGET_SENSITIVITY):
    candidates = np.unique(probabilities)[::-1]
    positives = targets == 1
    best_threshold, best_specificity = None, -1.0
    for threshold in candidates:
        predictions = probabilities >= threshold
        sensitivity = predictions[positives].mean()
        if sensitivity < target_sensitivity:
            continue
        specificity = (~predictions[~positives]).mean()
        if specificity > best_specificity:
            best_threshold, best_specificity = float(threshold), float(specificity)
    return best_threshold

In [ ]:
print("Development cohort")
china = load_cohort(CHINA_PATH)
print("External cohort")
korea = load_cohort(KOREA_PATH)
_k_lab = np.round(np.asarray(korea["k_value"], dtype=float), 1)
_y_lab = np.asarray(korea["y"]).copy()
_n_before = int((_y_lab == 2).sum())
_y_lab[(_y_lab == 0) & (_k_lab <= 3.3)] = 2
_y_lab[(_y_lab == 2) & (_k_lab > 3.3)] = 0
korea["y"] = _y_lab
print(f"External hypokalemia (K <= 3.3), 60-min file: {_n_before} -> {int((_y_lab == 2).sum())}")

train_pool_mask = cohort_mask(china, TRAIN_EXCLUDE_FLAGS)
test_eval_mask = cohort_mask(china, TEST_EXCLUDE_FLAGS)
external_mask = cohort_mask(korea, EXTERNAL_EXCLUDE_FLAGS) & (korea["y"] >= 0)
if "lab_dt_min" in korea:
    external_mask &= np.abs(np.asarray(korea["lab_dt_min"], dtype=float)) <= EXTERNAL_MAX_DT_MIN
if EXTERNAL_AGE_RANGE is not None:
    _age = np.asarray(korea["age_years"], dtype=float)
    external_mask &= (_age >= EXTERNAL_AGE_RANGE[0]) & (_age < EXTERNAL_AGE_RANGE[1])
print(f"External primary analysis: <= {EXTERNAL_MAX_DT_MIN} min, age {EXTERNAL_AGE_RANGE}")
for _name, _d in (("development", china), ("external", korea)):
    _vl = np.asarray(_d["_valid_len"])
    print(f"{_name} crop_valid_len: median {np.median(_vl):.0f}, < 400 samples {100 * np.mean(_vl < 400):.1f}%")

for _name, _d in (("development", china), ("external", korea)):
    _sex, _cnt = np.unique(np.asarray(_d["_meta"])[:, 0], return_counts=True)
    print(f"{_name} sex (0 male, 1 female, -1 missing): {dict(zip(_sex.astype(int).tolist(), _cnt.tolist()))}")

unique_patients, patient_labels = patient_level_label(china["ids"], china["y"])
cv_pts, test_pts, y_cv, _ = train_test_split(
    unique_patients, patient_labels, test_size=TEST_SIZE,
    stratify=patient_labels, random_state=SEED,
)
folds = list(StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED).split(cv_pts, y_cv))

split_payload = {"cv_pts": cv_pts, "test_pts": test_pts, "seed": np.asarray(SEED)}
for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    split_payload[f"train_pts_fold{fold}"] = cv_pts[train_idx]
    split_payload[f"val_pts_fold{fold}"] = cv_pts[val_idx]
np.savez(os.path.join(SAVE_DIR, "patient_split.npz"), **split_payload)

import hashlib


def _split_fingerprint():
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(np.asarray(china["ids"]).astype(str)).tobytes())
    h.update(np.ascontiguousarray(np.asarray(china["y"]).astype(np.int64)).tobytes())
    h.update(np.ascontiguousarray(np.sort(np.asarray(test_pts).astype(str))).tobytes())
    for _, _val_idx in folds:
        h.update(np.ascontiguousarray(np.sort(np.asarray(cv_pts[_val_idx]).astype(str))).tobytes())
    return h.hexdigest()[:16]


SPLIT_KEY = _split_fingerprint()

print(f"Split key: {SPLIT_KEY}")

test_rows = np.isin(china["ids"], test_pts) & test_eval_mask
print()
print(f"Patients: total {len(unique_patients)}, CV {len(cv_pts)}, test {len(test_pts)}")
print(f"Internal test: {int(test_rows.sum())} ECGs ({int((china['y'][test_rows] == 2).sum())} hypokalemia)")
print(f"External: {int(external_mask.sum())} ECGs ({int((korea['y'][external_mask] == 2).sum())} hypokalemia)")
print(f"Exclusion flags: train {TRAIN_EXCLUDE_FLAGS}, internal test {TEST_EXCLUDE_FLAGS}, external {EXTERNAL_EXCLUDE_FLAGS}")

In [ ]:
fold_rows = []
for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    val = np.isin(china["ids"], cv_pts[val_idx]) & train_pool_mask
    hypo = val & (china["y"] == 2)
    row = {"fold": fold, "ECG": int(val.sum()), "hypo_ECG": int(hypo.sum()),
           "hypo_patients": len(np.unique(china["ids"][hypo]))}
    if "age_years" in china:
        row["hypo_age_median"] = float(np.nanmedian(np.asarray(china["age_years"], dtype=float)[hypo]))
    for flag in ("icd_e876", "flag_stt_noise", "flag_tall_t"):
        if flag in china:
            row[f"hypo_with_{flag}"] = int((hypo & china[flag].astype(bool)).sum())
    fold_rows.append(row)
fold_composition = pd.DataFrame(fold_rows)
fold_composition.to_csv(os.path.join(SAVE_DIR, "fold_composition.csv"), index=False)
print(fold_composition.to_string(index=False))

In [ ]:
oof = dict(np.load(os.path.join(PRED_DIR, "oof_predictions.npz"), allow_pickle=True))
FINAL_THRESHOLD = select_threshold(oof["targets"], oof["probs"])
print(f"Threshold (out-of-fold, sensitivity >= {TARGET_SENSITIVITY}): {FINAL_THRESHOLD:.4f}")

predictions = {}
for name, rows in (("internal", test_rows), ("external", external_mask)):
    saved = dict(np.load(os.path.join(PRED_DIR, f"predictions_{name}.npz"), allow_pickle=True))
    predictions[name] = {k: saved[k] for k in ("rows", "y", "ids", "fold_probs", "prob")}
    if not np.array_equal(predictions[name]["rows"], np.where(rows)[0]):
        raise ValueError(f"Saved {name} predictions do not match the current cohort definition")
    print(f"{name}: {len(predictions[name]['y'])} ECGs, {int(predictions[name]['y'].sum())} hypokalemia")

In [ ]:
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LogisticRegression


def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def calibration_in_the_large(y, p, iterations=50):
    offset, a = logit(p), 0.0
    for _ in range(iterations):
        mu = 1 / (1 + np.exp(-(a + offset)))
        gradient, hessian = np.sum(y - mu), np.sum(mu * (1 - mu))
        if hessian <= 0:
            break
        step = gradient / hessian
        a += step
        if abs(step) < 1e-8:
            break
    return float(a)


def calibration_slope(y, p):
    x = logit(p).reshape(-1, 1)
    return float(LogisticRegression(C=1e10, max_iter=1000).fit(x, y).coef_[0, 0])


def all_metrics(y, p, threshold):
    pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    prevalence = y.mean()
    brier = brier_score_loss(y, p)
    reference_brier = prevalence * (1 - prevalence)
    return {
        "AUROC": roc_auc_score(y, p),
        "AUPRC": average_precision_score(y, p),
        "Prevalence": prevalence,
        "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "PPV": tp / (tp + fp) if tp + fp else np.nan,
        "NPV": tn / (tn + fn) if tn + fn else np.nan,
        "F1": f1_score(y, pred, zero_division=0),
        "Macro-F1": f1_score(y, pred, average="macro", zero_division=0),
        "Flagged (%)": 100 * pred.mean(),
        "Brier": brier,
        "Brier (prevalence only)": reference_brier,
        "Scaled Brier": 1 - brier / reference_brier if reference_brier else np.nan,
        "Calibration intercept": calibration_in_the_large(y, p),
        "Calibration slope": calibration_slope(y, p),
        "O/E": y.sum() / p.sum(),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    }


COUNT_METRICS = {"Sensitivity": ("TP", "FN"), "Specificity": ("TN", "FP"), "PPV": ("TP", "FP"), "NPV": ("TN", "FN")}
NO_CI = ("TP", "FP", "TN", "FN", "Prevalence", "Brier (prevalence only)")


def exact_ci(k, n, alpha=0.05):
    if n == 0:
        return np.nan, np.nan
    low = beta_dist.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    high = beta_dist.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return float(low), float(high)


def patient_bootstrap(y, p, ids, threshold, n_boot=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index_by_patient = pd.Series(np.arange(len(ids))).groupby(np.asarray(ids)).indices
    samples = []
    for _ in range(n_boot):
        chosen = rng.choice(patients, size=len(patients), replace=True)
        idx = np.concatenate([index_by_patient[pid] for pid in chosen])
        if y[idx].min() == y[idx].max():
            continue
        samples.append(all_metrics(y[idx], p[idx], threshold))
    return pd.DataFrame(samples)


def summarize(name, pred, threshold, version=""):
    point = all_metrics(pred["y"], pred["prob"], threshold)
    boot = patient_bootstrap(pred["y"], pred["prob"], pred["ids"], threshold)
    rows = []
    for key, value in point.items():
        row = {"cohort": name, "version": version, "metric": key, "value": value,
               "ci_low": np.nan, "ci_high": np.nan, "ci_method": ""}
        if key in ("TP", "FP", "TN", "FN"):
            row["value"] = int(value)
        elif key in COUNT_METRICS:
            numerator, other = COUNT_METRICS[key]
            row["ci_low"], row["ci_high"] = exact_ci(int(point[numerator]), int(point[numerator] + point[other]))
            row["ci_method"] = "Clopper-Pearson"
        elif key not in NO_CI:
            row["ci_low"], row["ci_high"] = np.nanpercentile(boot[key], [2.5, 97.5])
            row["ci_method"] = "patient bootstrap"
        rows.append(row)
    return pd.DataFrame(rows)


def fold_summary(name, pred, version=""):
    aurocs = [roc_auc_score(pred["y"], p) for p in pred["fold_probs"]]
    auprcs = [average_precision_score(pred["y"], p) for p in pred["fold_probs"]]
    return {"cohort": name, "version": version,
            "AUROC mean (SD)": f"{np.mean(aurocs):.3f} ({np.std(aurocs, ddof=1):.3f})",
            "AUPRC mean (SD)": f"{np.mean(auprcs):.3f} ({np.std(auprcs, ddof=1):.3f})"}


def format_ci(table, keys=("AUROC", "AUPRC", "Sensitivity", "Specificity", "PPV", "NPV")):
    out = {}
    for _, r in table.iterrows():
        if r["metric"] in keys:
            out[r["metric"]] = f"{r['value']:.3f} ({r['ci_low']:.3f}–{r['ci_high']:.3f})"
    return out

In [ ]:
def fit_platt(y, p):
    lr = LogisticRegression(C=1e10, max_iter=1000).fit(logit(p).reshape(-1, 1), y)
    return float(lr.intercept_[0]), float(lr.coef_[0, 0])


def apply_platt(p, a, b):
    return 1 / (1 + np.exp(-(a + b * logit(p))))


platt = {}
oof_cal = np.zeros(len(oof["probs"]), dtype=float)
for fold in range(1, K_FOLDS + 1):
    m = oof["fold"] == fold
    a, b = fit_platt(oof["targets"][m], oof["probs"][m])
    platt[fold] = (a, b)
    oof_cal[m] = apply_platt(oof["probs"][m], a, b)
    print(f"fold{fold}: a = {a:.3f}, b = {b:.3f}")

CAL_THRESHOLD = select_threshold(oof["targets"], oof_cal)
print(f"\nThreshold before recalibration {FINAL_THRESHOLD:.4f}, after {CAL_THRESHOLD:.4f} "
      f"({CAL_THRESHOLD / oof['targets'].mean():.1f} x out-of-fold prevalence {oof['targets'].mean():.4f})")

cal_predictions = {}
for name in ("internal", "external"):
    fp = predictions[name]["fold_probs"]
    cal_fold = np.stack([apply_platt(fp[f - 1], *platt[f]) for f in range(1, K_FOLDS + 1)])
    cal_predictions[name] = {**predictions[name], "fold_probs": cal_fold, "prob": cal_fold.mean(axis=0)}
    np.savez(os.path.join(SAVE_DIR, f"predictions_{name}_recal.npz"),
             **cal_predictions[name], threshold=CAL_THRESHOLD)

MAIN_PRED = cal_predictions if USE_RECALIBRATED else predictions
MAIN_THRESHOLD = CAL_THRESHOLD if USE_RECALIBRATED else FINAL_THRESHOLD
MAIN_VERSION = "recalibrated" if USE_RECALIBRATED else "raw"
print(f"Main analysis: {MAIN_VERSION}, threshold {MAIN_THRESHOLD:.4f}")

In [ ]:
main_table = pd.concat([
    summarize("OOF", {"y": oof["targets"], "prob": oof_cal, "ids": china["ids"][oof["rows"]]}, CAL_THRESHOLD, "recalibrated"),
    *[summarize(n, predictions[n], FINAL_THRESHOLD, "raw") for n in ("internal", "external")],
    *[summarize(n, cal_predictions[n], CAL_THRESHOLD, "recalibrated") for n in ("internal", "external")],
])
main_table.to_csv(os.path.join(SAVE_DIR, "main_results.csv"), index=False)

fold_table = pd.DataFrame([fold_summary(n, predictions[n], "raw") for n in ("internal", "external")])
fold_table.to_csv(os.path.join(SAVE_DIR, "fold_summary.csv"), index=False)



pd.set_option("display.float_format", lambda v: f"{v:.3f}")
print(main_table.to_string(index=False))
print(fold_table.to_string(index=False))

In [ ]:
keys = ["AUROC", "AUPRC", "Sensitivity", "Specificity", "Flagged (%)", "TP", "FP", "TN", "FN"]
check_rows = []
y_int = predictions["internal"]["y"]
for version, oof_p, preds, th in (("raw", oof["probs"], predictions, FINAL_THRESHOLD),
                                  ("recalibrated", oof_cal, cal_predictions, CAL_THRESHOLD)):
    for fold in range(1, K_FOLDS + 1):
        m = oof["fold"] == fold
        check_rows.append({"version": version, "set": f"OOF fold{fold}", "n_hypo": int(oof["targets"][m].sum()),
                           **{k: all_metrics(oof["targets"][m], oof_p[m], th)[k] for k in keys}})
        check_rows.append({"version": version, "set": f"test, fold {fold} only", "n_hypo": int(y_int.sum()),
                           **{k: all_metrics(y_int, preds["internal"]["fold_probs"][fold - 1], th)[k] for k in keys}})
    check_rows.append({"version": version, "set": "test, ensemble", "n_hypo": int(y_int.sum()),
                       **{k: all_metrics(y_int, preds["internal"]["prob"], th)[k] for k in keys}})
check_table = pd.DataFrame(check_rows)
check_table.to_csv(os.path.join(SAVE_DIR, "appendix_fold_consistency.csv"), index=False)
print(check_table.round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

SHOW_RAW = False
CLASS_LABELS = ["Non-hypokalemia", "Hypokalemia"]
CMAP = plt.get_cmap("Blues")


def plot_cm(ax, y, p, threshold, title):
    pred = (p >= threshold).astype(int)
    cm = confusion_matrix(y, pred, labels=[0, 1])
    row_pct = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    ax.imshow(row_pct, cmap=CMAP, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ink = "white" if row_pct[i, j] > 0.6 else "#1f1f1f"
            ax.text(j, i, f"{cm[i, j]:,}\n({100 * row_pct[i, j]:.1f}%)",
                    ha="center", va="center", fontsize=12, color=ink)
    ax.set_xticks([0, 1], CLASS_LABELS)
    ax.set_yticks([0, 1], CLASS_LABELS, rotation=90, va="center")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0.5], minor=True)
    ax.set_yticks([0.5], minor=True)
    ax.grid(which="minor", color="white", linewidth=3)
    ax.tick_params(which="both", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    ax.set_title(f"{title}\nn = {len(y):,}, hypokalemia {int(y.sum())}\n"
                 f"Sens {sens:.3f} · Spec {spec:.3f} · PPV {ppv:.3f}", fontsize=11)


EXT_LABEL = f"External ({EXTERNAL_AGE_RANGE[0]}–{EXTERNAL_AGE_RANGE[1] - 1} y, ≤{EXTERNAL_MAX_DT_MIN} min)" \
    if EXTERNAL_AGE_RANGE is not None else f"External (≤{EXTERNAL_MAX_DT_MIN} min)"
cm_rows = [("Recalibrated" if USE_RECALIBRATED else "Raw", MAIN_PRED, MAIN_THRESHOLD)]
if SHOW_RAW and USE_RECALIBRATED:
    cm_rows.append(("Raw", predictions, FINAL_THRESHOLD))

fig, axes = plt.subplots(len(cm_rows), 2, figsize=(10, 4.9 * len(cm_rows)), squeeze=False)
for r, (version, preds, th) in enumerate(cm_rows):
    for c, (name, label) in enumerate((("internal", "Internal test"), ("external", EXT_LABEL))):
        plot_cm(axes[r, c], preds[name]["y"], preds[name]["prob"], th, f"{label} — {version}")
fig.text(0.5, 0.005, f"Threshold {MAIN_THRESHOLD:.4f}, selected on out-of-fold predictions (sensitivity ≥ "
         f"{TARGET_SENSITIVITY:.2f}) and applied unchanged. Cell color = proportion within the actual class.",
         ha="center", fontsize=9, color="#555555")
fig.tight_layout(rect=(0, 0.03, 1, 1), h_pad=3)
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
def exclusion_row(label, y, p, ids, keep):
    table = summarize(label, {"y": y[keep], "prob": p[keep], "ids": ids[keep]}, MAIN_THRESHOLD)
    return {"analysis": label, "n_ECG": int(keep.sum()), "n_patients": len(np.unique(ids[keep])),
            "n_hypo": int(y[keep].sum()), **format_ci(table, ("AUROC", "AUPRC"))}


exclusion_rows = []
for name, data, flag_name in (("internal", china, "flag_stt_noise"), ("external", korea, "flag_any_statement")):
    if flag_name not in data:
        print(f"{name}: {flag_name} not available, skipped")
        continue
    pr = MAIN_PRED[name]
    y, p, ids = pr["y"], pr["prob"], pr["ids"]
    flag = np.asarray(data[flag_name])[pr["rows"]].astype(bool)
    normal = np.asarray(data["y"])[pr["rows"]] == 0
    exclusion_rows.append(exclusion_row(f"{name}: no exclusion (primary)", y, p, ids, np.ones(len(y), dtype=bool)))
    exclusion_rows.append(exclusion_row(f"{name}: excluded from normal class only (original approach)", y, p, ids, ~(flag & normal)))
    exclusion_rows.append(exclusion_row(f"{name}: excluded from all classes", y, p, ids, ~flag))
exclusion_table = pd.DataFrame(exclusion_rows)
exclusion_table.to_csv(os.path.join(SAVE_DIR, "appendix_exclusion_comparison.csv"), index=False)
print(exclusion_table.to_string(index=False))

In [ ]:
ext = MAIN_PRED["external"]
ext_rows = ext["rows"]
k_value = np.round(np.asarray(korea["k_value"], dtype=float)[ext_rows], 1) if "k_value" in korea else None
sensitivity_rows = []


def add_result(label, y, p, ids):
    if len(np.unique(y)) < 2:
        sensitivity_rows.append({"analysis": label, "n_ECG": len(y), "n_hypo": int(y.sum()), "note": "single class only"})
        return
    table = summarize(label, {"y": y, "prob": p, "ids": ids}, MAIN_THRESHOLD)
    sensitivity_rows.append({"analysis": label, "n_ECG": len(y), "n_hypo": int(y.sum()), **format_ci(table)})


if k_value is not None and not np.isnan(k_value).all():
    for label, rule in (("K ≤ 3.5", lambda k: k <= 3.5),
                        ("K ≤ 3.3 (primary)", lambda k: k <= 3.3),
                        ("K ≤ 3.0", lambda k: k <= 3.0)):
        add_result(f"External: {label}", rule(k_value).astype(int), ext["prob"], ext["ids"])

    flagged = ext["prob"] >= MAIN_THRESHOLD
    severity = []
    for label, low, high in (("≤2.5", -np.inf, 2.5), ("2.6–3.0", 2.5, 3.0), ("3.1–3.3", 3.0, 3.3), ("3.4–3.5", 3.3, 3.5)):
        in_band = (k_value > low) & (k_value <= high)
        n_band = int(in_band.sum())
        detected = int((flagged & in_band).sum())
        low_ci, high_ci = exact_ci(detected, n_band)
        severity.append({"K band": label, "n": n_band, "detected": detected,
                         "sensitivity": detected / n_band if n_band else np.nan,
                         "ci_low": low_ci, "ci_high": high_ci})
    severity_table = pd.DataFrame(severity)
    severity_table.to_csv(os.path.join(SAVE_DIR, "external_severity.csv"), index=False)
    print(severity_table.round(3).to_string(index=False))
    fn_k = np.sort(k_value[(ext["y"] == 1) & ~flagged])
    print("Potassium of false-negative ECGs:", fn_k.tolist())

    rho, pval = spearmanr(ext["prob"], k_value, nan_policy="omit")
    print(f"Spearman (predicted risk, K) = {rho:.3f} (P = {pval:.3g})")

age = np.asarray(korea["age_years"], dtype=float)[ext_rows]
if not np.isnan(age).all():
    m = (age >= 1) & (age < 15)
    add_result("External: 1–14 y (age-matched)", ext["y"][m], ext["prob"][m], ext["ids"][m])
    m = age >= 15
    add_result("External: ≥15 y", ext["y"][m], ext["prob"][m], ext["ids"][m])

if "flag_any_statement" in korea:
    m = ~korea["flag_any_statement"][ext_rows].astype(bool)
    add_result("External: statements excluded (all classes)", ext["y"][m], ext["prob"][m], ext["ids"][m])

sex = korea["_meta"][ext_rows, 0]
age_cat = korea["_meta"][ext_rows, 1]
for label, m in (("male", sex == 0), ("female", sex == 1)):
    add_result(f"External subgroup: {label}", ext["y"][m], ext["prob"][m], ext["ids"][m])
for code, label in enumerate(("1–6 y", "7–9 y", "10–12 y", "≥13 y")):
    m = age_cat == code
    add_result(f"External subgroup: {label}", ext["y"][m], ext["prob"][m], ext["ids"][m])
for label, (a0, a1) in (("13–14 y", (13, 15)), ("15–17 y (outside development range)", (15, 18))):
    m = (age >= a0) & (age < a1)
    add_result(f"External subgroup: {label}", ext["y"][m], ext["prob"][m], ext["ids"][m])

sensitivity_table = pd.DataFrame(sensitivity_rows)
sensitivity_table.to_csv(os.path.join(SAVE_DIR, "sensitivity_subgroups.csv"), index=False)
print(sensitivity_table.to_string(index=False))

In [ ]:
GRID_WINDOWS = [5, 10, 30, 60]
GRID_K = {"K ≤ 3.0": 3.0, "K ≤ 3.3": 3.3, "K ≤ 3.5": 3.5}
GRID_AGE = {"1–14 y": (1, 15), "1–17 y": (1, 18)}
GRID_BOOT = 500

all_ext = korea["_ok"] & (np.asarray(korea["y"]) >= 0)
_saved_all = dict(np.load(os.path.join(PRED_DIR, "predictions_external_all.npz"), allow_pickle=True))
if not np.array_equal(_saved_all["rows"], np.where(all_ext)[0]):
    raise ValueError("Saved external predictions do not match the external cohort file")
p_all = np.mean(_saved_all["fold_probs_recal"], axis=0)

rows_all = np.where(all_ext)[0]
dt_all = np.abs(np.asarray(korea["lab_dt_min"], dtype=float))[rows_all]
k_all = np.round(np.asarray(korea["k_value"], dtype=float), 1)[rows_all]
age_all = np.asarray(korea["age_years"], dtype=float)[rows_all]
ids_all = np.asarray(korea["ids"])[rows_all]
print(f"External ECGs: {len(rows_all)}, maximum time difference {np.nanmax(dt_all):.0f} min")
if np.nanmax(dt_all) < max(GRID_WINDOWS) * 0.5:
    print("Maximum time difference is shorter than expected; check PAIR_WINDOW_MIN in the data rebuild notebook.")


def auc_ci(y, p, ids, n_boot=GRID_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index = pd.Series(np.arange(len(ids))).groupby(ids).indices
    aurocs, auprcs = [], []
    for _ in range(n_boot):
        idx = np.concatenate([index[q] for q in rng.choice(patients, len(patients), replace=True)])
        if y[idx].min() == y[idx].max():
            continue
        aurocs.append(roc_auc_score(y[idx], p[idx]))
        auprcs.append(average_precision_score(y[idx], p[idx]))
    return np.percentile(aurocs, [2.5, 97.5]), np.percentile(auprcs, [2.5, 97.5])


def fmt(v, lo, hi):
    return f"{v:.3f} ({lo:.3f}–{hi:.3f})"


grid_rows = []
for window in GRID_WINDOWS:
    for age_label, (a0, a1) in GRID_AGE.items():
        m = (dt_all <= window) & (age_all >= a0) & (age_all < a1)
        for k_label, cut in GRID_K.items():
            y = (k_all[m] <= cut).astype(int)
            p, ids = p_all[m], ids_all[m]
            row = {"window (min)": window, "age": age_label, "K cutoff": k_label,
                   "n_ECG": int(m.sum()), "n_patients": len(np.unique(ids)),
                   "n_hypo": int(y.sum()), "prevalence": y.mean() if len(y) else np.nan}
            if 2 <= y.sum() < len(y):
                pred = p >= MAIN_THRESHOLD
                tp, fn = int((pred & (y == 1)).sum()), int((~pred & (y == 1)).sum())
                tn, fp = int((~pred & (y == 0)).sum()), int((pred & (y == 0)).sum())
                (au_lo, au_hi), (ap_lo, ap_hi) = auc_ci(y, p, ids)
                row.update({
                    "AUROC": fmt(roc_auc_score(y, p), au_lo, au_hi),
                    "AUPRC": fmt(average_precision_score(y, p), ap_lo, ap_hi),
                    "Sensitivity": fmt(tp / (tp + fn), *exact_ci(tp, tp + fn)),
                    "Specificity": fmt(tn / (tn + fp), *exact_ci(tn, tn + fp)),
                    "PPV": fmt(tp / (tp + fp), *exact_ci(tp, tp + fp)) if tp + fp else "—",
                    "NPV": fmt(tn / (tn + fn), *exact_ci(tn, tn + fn)) if tn + fn else "—",
                    "Flagged (%)": round(100 * pred.mean(), 1),
                    "TP": tp, "FN": fn, "FP": fp, "TN": tn,
                })
            else:
                row["note"] = "fewer than 2 cases"
            is_primary = (window == EXTERNAL_MAX_DT_MIN and k_label == "K ≤ 3.3"
                          and EXTERNAL_AGE_RANGE is not None and (a0, a1) == tuple(EXTERNAL_AGE_RANGE))
            row["primary"] = "★" if is_primary else ""
            grid_rows.append(row)

grid = pd.DataFrame(grid_rows)
grid.to_csv(os.path.join(SAVE_DIR, "external_full_grid.csv"), index=False)

for age_label in GRID_AGE:
    sub = grid[grid["age"] == age_label].copy()
    sub["cell"] = sub.apply(lambda r: f"{r['AUROC']} [n={r['n_hypo']}]" if isinstance(r.get("AUROC"), str)
                            else f"— [n={r['n_hypo']}]", axis=1)
    pivot = sub.pivot(index="window (min)", columns="K cutoff", values="cell")[list(GRID_K)]
    print(f"\n=== External {age_label}: AUROC (95% CI) [cases] ===")
    print(pivot.to_string())
    pivot.to_csv(os.path.join(SAVE_DIR, f"external_grid_auroc_{age_label.replace(' ', '').replace('–', '-')}.csv"))

print("\n=== All results (★ = primary) ===")
print(grid.drop(columns=["prevalence"]).to_string(index=False))

In [ ]:
CM_CUTOFFS = {"K ≤ 3.0": 3.0, "K ≤ 3.3 (primary)": 3.3, "K ≤ 3.5": 3.5}

fig, axes = plt.subplots(len(GRID_AGE), len(CM_CUTOFFS),
                         figsize=(5 * len(CM_CUTOFFS), 4.9 * len(GRID_AGE)), squeeze=False)
for r, (age_label, (a0, a1)) in enumerate(GRID_AGE.items()):
    m = (dt_all <= EXTERNAL_MAX_DT_MIN) & (age_all >= a0) & (age_all < a1)
    for c, (k_label, cut) in enumerate(CM_CUTOFFS.items()):
        y_k = (k_all[m] <= cut).astype(int)
        plot_cm(axes[r, c], y_k, p_all[m], MAIN_THRESHOLD, f"External {age_label}, {k_label}")
        axes[r, c].set_yticklabels([f"K > {cut}", f"K ≤ {cut}"], rotation=90, va="center")
fig.text(0.5, 0.005,
         f"ECG–potassium interval ≤ {EXTERNAL_MAX_DT_MIN} min. Same model, recalibration, and threshold "
         f"({MAIN_THRESHOLD:.4f}) in all panels; only the potassium cutoff changes. "
         "Cell color = proportion within the actual class.",
         ha="center", fontsize=9, color="#555555")
fig.tight_layout(rect=(0, 0.03, 1, 1), h_pad=3)
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix_by_k_cutoff.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrix_by_k_cutoff.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
RING_CUT = 3.3
TEST_KEY = None

dt_signed = np.asarray(korea["lab_dt_min"], dtype=float)[rows_all]
base = (age_all >= EXTERNAL_AGE_RANGE[0]) & (age_all < EXTERNAL_AGE_RANGE[1])
y_ring = (k_all <= RING_CUT).astype(int)
in5 = base & (dt_all <= 5)
in5_10 = base & (dt_all > 5) & (dt_all <= 10)
in10 = base & (dt_all <= 10)
test_all = np.asarray(korea[TEST_KEY])[rows_all] if TEST_KEY else None


def auc_or_nan(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan


ring_rows = []
for name, m in (("≤5 min", in5), ("5–10 min only", in5_10), ("≤10 min", in10)):
    y, p = y_ring[m], p_all[m]
    pred = p >= MAIN_THRESHOLD
    k_h = k_all[m][y == 1]
    ring_rows.append({
        "set": name, "n_ECG": int(m.sum()), "n_patients": len(np.unique(ids_all[m])), "n_hypo": int(y.sum()),
        "AUROC": round(auc_or_nan(y, p), 3),
        "Sens": round(pred[y == 1].mean(), 3) if y.sum() else np.nan,
        "Spec": round((~pred[y == 0]).mean(), 3),
        "hypo K median (min–max)": f"{np.median(k_h):.1f} ({k_h.min():.1f}–{k_h.max():.1f})" if len(k_h) else "",
        "hypo K ≤3.0 (n)": int((k_h <= 3.0).sum()),
        "hypo K 3.1–3.3 (n)": int(((k_h > 3.0) & (k_h <= 3.3)).sum()),
        "score median, hypo": round(np.median(p[y == 1]), 4) if y.sum() else np.nan,
        "score median, non-hypo": round(np.median(p[y == 0]), 4),
        "lab after ECG (%)": round(100 * (dt_signed[m] > 0).mean(), 1),
    })
ring_table = pd.DataFrame(ring_rows)
pd.set_option("display.width", 260)
print(f"=== {AGE_TXT}, K ≤ 3.3: by window ===")
print(ring_table.to_string(index=False))

print("\n=== Cross-window AUROC ===")
for label, pos_m, neg_m in (("≤5 hypo  vs ≤5 non-hypo", in5, in5),
                            ("≤5 hypo  vs 5–10 non-hypo", in5, in5_10),
                            ("5–10 hypo vs ≤5 non-hypo", in5_10, in5),
                            ("5–10 hypo vs 5–10 non-hypo", in5_10, in5_10)):
    pos = p_all[pos_m & (y_ring == 1)]
    neg = p_all[neg_m & (y_ring == 0)]
    yy = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    print(f"{label}: n_hypo {len(pos)}, n_non-hypo {len(neg)}, AUROC {auc_or_nan(yy, np.r_[pos, neg]):.3f}")

neg10 = p_all[in10 & (y_ring == 0)]
case_rows = []
for i in np.where(in10 & (y_ring == 1))[0]:
    row = {"window": "≤5" if dt_all[i] <= 5 else "5–10", "patient": ids_all[i], "age": round(age_all[i], 1),
           "K": k_all[i], "dt (min, signed)": round(dt_signed[i], 1), "score": round(p_all[i], 4),
           "flagged": bool(p_all[i] >= MAIN_THRESHOLD),
           "percentile vs ≤10 non-hypo": round(100 * (neg10 < p_all[i]).mean(), 1)}
    if test_all is not None:
        row["test"] = test_all[i]
    case_rows.append(row)
case_table = pd.DataFrame(case_rows).sort_values(["window", "K"])
print(f"\nHypokalemia ECGs within 10 min: {len(case_table)} (per-ECG table saved to window_5_10_hypo_cases.csv)")
n_pat_both = len(set(case_table.loc[case_table["window"] == "≤5", "patient"])
                 & set(case_table.loc[case_table["window"] == "5–10", "patient"]))
print(f"\nPatients with hypokalemia ECGs in both windows: {n_pat_both}")

if test_all is not None:
    print(f"\n=== By potassium test type ({AGE_TXT}) ===")
    for name, m in (("≤5 min", in5), ("5–10 min only", in5_10)):
        print(f"\n[{name}]")
        print(pd.crosstab(test_all[m], y_ring[m], margins=True).rename(columns={0: "K >3.3", 1: "K ≤3.3"}))

ring_table.to_csv(os.path.join(SAVE_DIR, "window_5_10_summary.csv"), index=False)
case_table.drop(columns=["patient"]).to_csv(os.path.join(SAVE_DIR, "window_5_10_hypo_cases.csv"), index=False)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

LR_BOOT = 1000
LR_AGE_MODE = "category"
LR_TAG = "_agecat" if LR_AGE_MODE == "category" else ""
print(f"LR age input: {LR_AGE_MODE}")


def age_columns(data, rows, meta):
    if LR_AGE_MODE == "category":
        cat = meta[:, 1]
        cols = [np.where(cat < 0, np.nan, (cat == c).astype(float)) for c in (1, 2, 3)]
        return np.column_stack(cols), ["age 7–9 y", "age 10–12 y", "age ≥13 y"]
    age = np.asarray(data["age_years"], dtype=float)[rows] if "age_years" in data else np.full(len(rows), np.nan)
    if np.isnan(age).all():
        age = meta[:, 1]
    return age.reshape(-1, 1), ["age"]


def lr_features(data, rows):
    meta = np.asarray(data["_meta"], dtype=float)[rows]
    age_cols, _ = age_columns(data, rows, meta)
    sex = np.where(meta[:, 0] < 0, np.nan, meta[:, 0])
    return {
        "demo": np.column_stack([age_cols, sex]),
        "demo_ecg": np.column_stack([age_cols, sex, meta[:, 2], meta[:, 3], meta[:, 4]]),
    }


def make_lr():
    return make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                         StandardScaler(), LogisticRegression(max_iter=2000))


LR_SPECS = {
    "LR: age + sex": "demo",
    "LR: age + sex + HR + energy ratio + slope": "demo_ecg",
}
_AGE_NAMES = ["age 7–9 y", "age 10–12 y", "age ≥13 y"] if LR_AGE_MODE == "category" else ["age"]
LR_FEATURE_NAMES = {"demo": _AGE_NAMES + ["sex"],
                    "demo_ecg": _AGE_NAMES + ["sex", "heart rate", "energy ratio", "slope"]}

y_china_bin = (np.asarray(china["y"]) == 2).astype(int)
int_rows = MAIN_PRED["internal"]["rows"]
ext_rows = MAIN_PRED["external"]["rows"]
lr_results = {}

for label, key in LR_SPECS.items():
    oof_p = np.full(len(y_china_bin), np.nan)
    models = []
    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        train_rows = np.where(np.isin(china["ids"], cv_pts[train_idx]) & train_pool_mask)[0]
        val_rows = np.where(np.isin(china["ids"], cv_pts[val_idx]) & train_pool_mask)[0]
        model_lr = make_lr().fit(lr_features(china, train_rows)[key], y_china_bin[train_rows])
        oof_p[val_rows] = model_lr.predict_proba(lr_features(china, val_rows)[key])[:, 1]
        models.append(model_lr)
    has_oof = ~np.isnan(oof_p)
    th = select_threshold(y_china_bin[has_oof], oof_p[has_oof])
    p_int = np.mean([m.predict_proba(lr_features(china, int_rows)[key])[:, 1] for m in models], axis=0)
    p_ext = np.mean([m.predict_proba(lr_features(korea, ext_rows)[key])[:, 1] for m in models], axis=0)
    lr_results[label] = {"threshold": th, "internal": p_int, "external": p_ext}
    print(f"{label} | OOF threshold {th:.4f}")


def paired_delta_auc(y, p_a, p_b, ids, n_boot=LR_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index = pd.Series(np.arange(len(ids))).groupby(ids).indices
    point = roc_auc_score(y, p_a) - roc_auc_score(y, p_b)
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([index[q] for q in rng.choice(patients, len(patients), replace=True)])
        if y[idx].min() == y[idx].max():
            continue
        diffs.append(roc_auc_score(y[idx], p_a[idx]) - roc_auc_score(y[idx], p_b[idx]))
    diffs = np.asarray(diffs)
    p_value = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return point, *np.percentile(diffs, [2.5, 97.5]), p_value


compare_rows = []
for cohort, key_name in (("Internal test", "internal"), (EXT_LABEL, "external")):
    pr = MAIN_PRED[key_name]
    y, ids = pr["y"], pr["ids"]
    candidates = [("Deep learning (ensemble)", pr["prob"], MAIN_THRESHOLD)]
    candidates += [(label, lr_results[label][key_name], lr_results[label]["threshold"]) for label in LR_SPECS]
    for name, p, th in candidates:
        table = summarize(name, {"y": y, "prob": p, "ids": ids}, th)
        row = {"cohort": cohort, "model": name, "n_ECG": len(y), "n_hypo": int(y.sum()),
               "threshold": round(th, 4), **format_ci(table)}
        flagged = table.loc[table["metric"] == "Flagged (%)", "value"]
        row["Flagged (%)"] = round(float(flagged.iloc[0]), 1) if len(flagged) else np.nan
        if name != "Deep learning (ensemble)":
            d, lo, hi, pv = paired_delta_auc(y, pr["prob"], p, ids)
            row["ΔAUROC (DL − LR)"] = f"{d:+.3f} ({lo:+.3f} to {hi:+.3f}), P = {pv:.3f}"
        compare_rows.append(row)

compare_table = pd.DataFrame(compare_rows)
compare_table.to_csv(os.path.join(SAVE_DIR, f"comparison_logistic_regression{LR_TAG}.csv"), index=False)
print("\n" + compare_table.to_string(index=False))

cv_rows = np.where(np.isin(china["ids"], cv_pts) & train_pool_mask)[0]
X_full = lr_features(china, cv_rows)["demo_ecg"]
names = LR_FEATURE_NAMES["demo_ecg"]
y_cv = y_china_bin[cv_rows]
or_rows = []
m_all = make_lr().fit(X_full, y_cv)
for j, name in enumerate(names):
    if np.isnan(X_full[:, j]).all():
        or_rows.append({"variable": name, "univariable OR per SD": "all missing", "multivariable OR per SD": "all missing"})
        continue
    m_uni = make_lr().fit(X_full[:, [j]], y_cv)
    or_rows.append({"variable": name,
                    "univariable OR per SD": round(float(np.exp(m_uni[-1].coef_[0, 0])), 2),
                    "multivariable OR per SD": round(float(np.exp(m_all[-1].coef_[0, j])), 2)})
or_table = pd.DataFrame(or_rows)
or_table.to_csv(os.path.join(SAVE_DIR, f"logistic_regression_odds_ratios{LR_TAG}.csv"), index=False)
print("\n" + or_table.to_string(index=False))
print("\nSpearman correlation between predictors")
print(pd.DataFrame(X_full, columns=names).corr(method="spearman").round(2).to_string())

In [ ]:
def run_lr_years():
    global LR_AGE_MODE
    saved_mode, LR_AGE_MODE = LR_AGE_MODE, "years"
    try:
        y_bin = (np.asarray(china["y"]) == 2).astype(int)
        rows_int, rows_ext = MAIN_PRED["internal"]["rows"], MAIN_PRED["external"]["rows"]
        results = {}
        for label, key in LR_SPECS.items():
            oof_p = np.full(len(y_bin), np.nan)
            fold_models = []
            for train_idx, val_idx in folds:
                tr = np.where(np.isin(china["ids"], cv_pts[train_idx]) & train_pool_mask)[0]
                va = np.where(np.isin(china["ids"], cv_pts[val_idx]) & train_pool_mask)[0]
                fitted = make_lr().fit(lr_features(china, tr)[key], y_bin[tr])
                oof_p[va] = fitted.predict_proba(lr_features(china, va)[key])[:, 1]
                fold_models.append(fitted)
            has_oof = ~np.isnan(oof_p)
            results[label] = {
                "threshold": select_threshold(y_bin[has_oof], oof_p[has_oof]),
                "internal": np.mean([f.predict_proba(lr_features(china, rows_int)[key])[:, 1] for f in fold_models], axis=0),
                "external": np.mean([f.predict_proba(lr_features(korea, rows_ext)[key])[:, 1] for f in fold_models], axis=0),
            }
            print(f"{label} [age in years] | OOF threshold {results[label]['threshold']:.4f}")

        cv_idx = np.where(np.isin(china["ids"], cv_pts) & train_pool_mask)[0]
        X = lr_features(china, cv_idx)["demo_ecg"]
        names = ["age (years)", "sex", "heart rate", "energy ratio", "slope"]
        full_fit = make_lr().fit(X, y_bin[cv_idx])
        or_rows = []
        for j, name in enumerate(names):
            uni = make_lr().fit(X[:, [j]], y_bin[cv_idx])
            or_rows.append({"variable": name,
                            "univariable OR per SD": round(float(np.exp(uni[-1].coef_[0, 0])), 2),
                            "multivariable OR per SD": round(float(np.exp(full_fit[-1].coef_[0, j])), 2)})
        return results, pd.DataFrame(or_rows)
    finally:
        LR_AGE_MODE = saved_mode


def lr_years_comparison(results_years):
    out = []
    for cohort, key_name in (("Internal test", "internal"), (EXT_LABEL, "external")):
        pr = MAIN_PRED[key_name]
        y, ids = pr["y"], pr["ids"]
        for label in LR_SPECS:
            p_cat, p_yr = lr_results[label][key_name], results_years[label][key_name]
            th_yr = results_years[label]["threshold"]
            table = summarize(label, {"y": y, "prob": p_yr, "ids": ids}, th_yr)
            row = {"cohort": cohort, "model": f"{label} [age in years]", "n_ECG": len(y), "n_hypo": int(y.sum()),
                   "threshold": round(th_yr, 4), **format_ci(table)}
            d, lo, hi, pv = paired_delta_auc(y, pr["prob"], p_yr, ids)
            row["ΔAUROC (DL − LR years)"] = f"{d:+.3f} ({lo:+.3f} to {hi:+.3f}), P = {pv:.3f}"
            d, lo, hi, pv = paired_delta_auc(y, p_yr, p_cat, ids)
            row["ΔAUROC (LR years − LR category)"] = f"{d:+.3f} ({lo:+.3f} to {hi:+.3f}), P = {pv:.3f}"
            out.append(row)
    return pd.DataFrame(out)


lr_results_years, lr_or_years = run_lr_years()
lr_compare_years = lr_years_comparison(lr_results_years)
lr_compare_years.to_csv(os.path.join(SAVE_DIR, "comparison_logistic_regression_ageyears.csv"), index=False)
lr_or_years.to_csv(os.path.join(SAVE_DIR, "logistic_regression_odds_ratios_ageyears.csv"), index=False)
print("\n" + lr_compare_years.to_string(index=False))
print("\n" + lr_or_years.to_string(index=False))
print(f"\nLR_AGE_MODE = {LR_AGE_MODE}")

In [ ]:
from scipy.stats import binomtest

CMP_LABEL = "LR: age + sex + HR + energy ratio + slope"
CMP_KEY = LR_SPECS[CMP_LABEL]
CMP_K = 3.3
CMP_BOOT = 1000

lr_models = []
for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    train_rows = np.where(np.isin(china["ids"], cv_pts[train_idx]) & train_pool_mask)[0]
    lr_models.append(make_lr().fit(lr_features(china, train_rows)[CMP_KEY], y_china_bin[train_rows]))
lr_th = lr_results[CMP_LABEL]["threshold"]
lr_p_all = np.mean([m.predict_proba(lr_features(korea, rows_all)[CMP_KEY])[:, 1] for m in lr_models], axis=0)

_m = (dt_all <= EXTERNAL_MAX_DT_MIN) & (age_all >= EXTERNAL_AGE_RANGE[0]) & (age_all < EXTERNAL_AGE_RANGE[1])
print(f"External 5 min {AGE_TXT} LR AUROC = {roc_auc_score((k_all[_m] <= CMP_K).astype(int), lr_p_all[_m]):.3f} "
      f"(main LR: {roc_auc_score(MAIN_PRED['external']['y'], lr_results[CMP_LABEL]['external']):.3f})")


def paired_delta(y, p_a, p_b, ids, metric, n_boot=CMP_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index = pd.Series(np.arange(len(ids))).groupby(ids).indices
    point = metric(y, p_a) - metric(y, p_b)
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([index[q] for q in rng.choice(patients, len(patients), replace=True)])
        if y[idx].min() == y[idx].max():
            continue
        diffs.append(metric(y[idx], p_a[idx]) - metric(y[idx], p_b[idx]))
    diffs = np.asarray(diffs)
    p_value = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return f"{point:+.3f} ({lo:+.3f} to {hi:+.3f}), P = {p_value:.3f}"


def mcnemar_exact(correct_a, correct_b):
    b = int((correct_a & ~correct_b).sum())
    c = int((~correct_a & correct_b).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    return b, c, p


def compare_row(cohort, y, p_dl, p_lr, ids):
    pred_dl, pred_lr = p_dl >= MAIN_THRESHOLD, p_lr >= lr_th
    pos, neg = y == 1, y == 0
    b_sen, c_sen, p_sen = mcnemar_exact(pred_dl[pos], pred_lr[pos])
    b_spe, c_spe, p_spe = mcnemar_exact(~pred_dl[neg], ~pred_lr[neg])
    return {
        "cohort": cohort, "n_ECG": len(y), "n_hypo": int(y.sum()),
        "AUROC DL": round(roc_auc_score(y, p_dl), 3), "AUROC LR": round(roc_auc_score(y, p_lr), 3),
        "ΔAUROC (DL − LR)": paired_delta(y, p_dl, p_lr, ids, roc_auc_score),
        "AUPRC DL": round(average_precision_score(y, p_dl), 3), "AUPRC LR": round(average_precision_score(y, p_lr), 3),
        "ΔAUPRC (DL − LR)": paired_delta(y, p_dl, p_lr, ids, average_precision_score),
        "Sens DL": round(pred_dl[pos].mean(), 3), "Sens LR": round(pred_lr[pos].mean(), 3),
        "Sens McNemar": f"DL only {b_sen} / LR only {c_sen}, P = {p_sen:.3f}",
        "Spec DL": round((~pred_dl[neg]).mean(), 3), "Spec LR": round((~pred_lr[neg]).mean(), 3),
        "Spec McNemar": f"DL only {b_spe} / LR only {c_spe}, P = {p_spe:.3f}",
        "Flagged DL (%)": round(100 * pred_dl.mean(), 1), "Flagged LR (%)": round(100 * pred_lr.mean(), 1),
    }


cmp_rows = []
pr = MAIN_PRED["internal"]
cmp_rows.append(compare_row("Internal test", pr["y"], pr["prob"], lr_results[CMP_LABEL]["internal"], pr["ids"]))

for age_label, (a0, a1) in GRID_AGE.items():
    for window in GRID_WINDOWS:
        m = (dt_all <= window) & (age_all >= a0) & (age_all < a1)
        y = (k_all[m] <= CMP_K).astype(int)
        if 2 <= y.sum() < len(y):
            star = " ★" if (window == EXTERNAL_MAX_DT_MIN and (a0, a1) == tuple(EXTERNAL_AGE_RANGE)) else ""
            cmp_rows.append(compare_row(f"External {age_label}, ≤{window} min{star}",
                                        y, p_all[m], lr_p_all[m], ids_all[m]))

cmp_table = pd.DataFrame(cmp_rows)
cmp_table.to_csv(os.path.join(SAVE_DIR, "dl_vs_lr_by_window.csv"), index=False)
pd.set_option("display.width", 250)
print(f"\nK ≤ {CMP_K} | DL threshold {MAIN_THRESHOLD:.4f}, LR threshold {lr_th:.4f}")
print(cmp_table[["cohort", "n_ECG", "n_hypo", "AUROC DL", "AUROC LR", "ΔAUROC (DL − LR)"]].to_string(index=False))
print()
print(cmp_table[["cohort", "AUPRC DL", "AUPRC LR", "ΔAUPRC (DL − LR)"]].to_string(index=False))
print()
print(cmp_table[["cohort", "Sens DL", "Sens LR", "Sens McNemar", "Spec DL", "Spec LR", "Spec McNemar",
                 "Flagged DL (%)", "Flagged LR (%)"]].to_string(index=False))

In [ ]:
DCA_PT = np.linspace(0.005, 0.10, 96)
DCA_KEY_PT = [0.01, 0.02, 0.05]
DCA_BOOT = 1000
DCA_WIDE = 60

LR_ECG = "LR: age + sex + HR + energy ratio + slope"
LR_DEMO = "LR: age + sex"


def net_benefit(y, p, pt):
    n = len(y)
    pred = p >= pt
    tp = (pred & (y == 1)).sum()
    fp = (pred & (y == 0)).sum()
    return tp / n - fp / n * pt / (1 - pt)


def nb_treat_all(y, pt):
    prev = y.mean()
    return prev - (1 - prev) * pt / (1 - pt)


def nb_ci(y, p, ids, pt, n_boot=DCA_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index = pd.Series(np.arange(len(ids))).groupby(ids).indices
    nb, diff = [], []
    for _ in range(n_boot):
        idx = np.concatenate([index[q] for q in rng.choice(patients, len(patients), replace=True)])
        v = net_benefit(y[idx], p[idx], pt)
        nb.append(v)
        diff.append(v - max(nb_treat_all(y[idx], pt), 0))
    return np.percentile(nb, [2.5, 97.5]), np.percentile(diff, [2.5, 97.5])


pr_int, pr_ext = MAIN_PRED["internal"], MAIN_PRED["external"]
m_wide = (dt_all <= DCA_WIDE) & (age_all >= EXTERNAL_AGE_RANGE[0]) & (age_all < EXTERNAL_AGE_RANGE[1])

lr_demo_models = []
for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    train_rows = np.where(np.isin(china["ids"], cv_pts[train_idx]) & train_pool_mask)[0]
    lr_demo_models.append(make_lr().fit(lr_features(china, train_rows)["demo"], y_china_bin[train_rows]))
lr_demo_p_all = np.mean([m.predict_proba(lr_features(korea, rows_all)["demo"])[:, 1] for m in lr_demo_models], axis=0)

DCA_SETS = {
    "Internal test": {
        "y": pr_int["y"], "ids": pr_int["ids"],
        "models": {"Deep learning": pr_int["prob"],
                   "LR: ECG features": lr_results[LR_ECG]["internal"],
                   "LR: age + sex": lr_results[LR_DEMO]["internal"]}},
    f"External {AGE_TXT}, ≤5 min (primary)": {
        "y": pr_ext["y"], "ids": pr_ext["ids"],
        "models": {"Deep learning": pr_ext["prob"],
                   "LR: ECG features": lr_results[LR_ECG]["external"],
                   "LR: age + sex": lr_results[LR_DEMO]["external"]}},
    f"External {AGE_TXT}, ≤{DCA_WIDE} min": {
        "y": (k_all[m_wide] <= 3.3).astype(int), "ids": ids_all[m_wide],
        "models": {"Deep learning": p_all[m_wide],
                   "LR: ECG features": lr_p_all[m_wide],
                   "LR: age + sex": lr_demo_p_all[m_wide]}},
}

DCA_STYLE = {"Deep learning": ("#c0392b", "-", 2.0),
             "LR: ECG features": ("#2c7fb8", "-", 1.6),
             "LR: age + sex": ("#7f7f7f", "-.", 1.2)}
OP_TH = {"Deep learning": MAIN_THRESHOLD, "LR: ECG features": lr_results[LR_ECG]["threshold"]}

fig, axes = plt.subplots(1, len(DCA_SETS), figsize=(6.2 * len(DCA_SETS), 5), squeeze=False)
dca_rows = []
for ax, (set_name, s) in zip(axes[0], DCA_SETS.items()):
    y, ids = s["y"], s["ids"]
    nb_all = np.array([nb_treat_all(y, pt) for pt in DCA_PT])
    ymax = y.mean() * 1.15
    ax.plot(DCA_PT * 100, nb_all, color="#333", lw=1, ls="--", label="Test all")
    ax.axhline(0, color="#333", lw=1, ls=":", label="Test none")
    for name, p in s["models"].items():
        c, ls, lw = DCA_STYLE[name]
        ax.plot(DCA_PT * 100, [net_benefit(y, p, pt) for pt in DCA_PT], color=c, ls=ls, lw=lw, label=name)
        if name in OP_TH:
            ax.axvline(OP_TH[name] * 100, color=c, lw=0.7, ls=":", alpha=0.8)
        for pt in DCA_KEY_PT:
            nb = net_benefit(y, p, pt)
            ref = max(nb_treat_all(y, pt), 0)
            (lo, hi), (dlo, dhi) = nb_ci(y, p, ids, pt)
            dca_rows.append({
                "cohort": set_name, "model": name, "threshold (%)": pt * 100,
                "n_ECG": len(y), "n_hypo": int(y.sum()), "prevalence (%)": round(100 * y.mean(), 2),
                "NB": f"{nb:.4f} ({lo:.4f} to {hi:.4f})",
                "NB − max(test all, none)": f"{nb - ref:+.4f} ({dlo:+.4f} to {dhi:+.4f})",
                "tests avoided /100 vs test all": round((nb - nb_treat_all(y, pt)) / (pt / (1 - pt)) * 100, 1),
            })
    ax.set_ylim(-ymax * 0.4, ymax)
    ax.set_xlim(DCA_PT[0] * 100, DCA_PT[-1] * 100)
    ax.set_xlabel("Threshold probability (%)")
    ax.set_ylabel("Net benefit")
    ax.set_title(f"{set_name}\n(n = {len(y)}, hypokalemia = {int(y.sum())}, {100 * y.mean():.1f}%)", fontsize=10)
    ax.grid(alpha=0.2)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.text(0.5, -0.11, "Dotted vertical lines: prespecified operating thresholds (deep learning, red; LR, blue). "
         "Net benefit = TP/n − FP/n × pt/(1 − pt).", ha="center", fontsize=9, color="#555")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "decision_curve.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(SAVE_DIR, "decision_curve.pdf"), bbox_inches="tight")
plt.show()

dca_table = pd.DataFrame(dca_rows)
dca_table.to_csv(os.path.join(SAVE_DIR, "decision_curve_table.csv"), index=False)
pd.set_option("display.width", 250)
print(dca_table.to_string(index=False))

In [ ]:
DNB_PT = sorted(set(DCA_KEY_PT + [round(MAIN_THRESHOLD, 4)]))
DNB_BOOT = 1000


def paired_delta_nb(y, p_a, p_b, ids, pt, n_boot=DNB_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    patients = np.unique(ids)
    index = pd.Series(np.arange(len(ids))).groupby(ids).indices
    point = net_benefit(y, p_a, pt) - net_benefit(y, p_b, pt)
    diffs = np.array([net_benefit(y[idx], p_a[idx], pt) - net_benefit(y[idx], p_b[idx], pt)
                      for idx in (np.concatenate([index[q] for q in rng.choice(patients, len(patients), replace=True)])
                                  for _ in range(n_boot))])
    p_value = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return point, lo, hi, p_value


dnb_rows = []
for set_name, s in DCA_SETS.items():
    y, ids, mdl = s["y"], s["ids"], s["models"]
    for comp in ["LR: ECG features", "LR: age + sex"]:
        for pt in DNB_PT:
            d, lo, hi, pv = paired_delta_nb(y, mdl["Deep learning"], mdl[comp], ids, pt)
            dnb_rows.append({"cohort": set_name, "comparison": f"DL − {comp}", "threshold (%)": round(pt * 100, 2),
                             "ΔNB": f"{d:+.4f} ({lo:+.4f} to {hi:+.4f})",
                             "net true positives /100": f"{100 * d:+.2f} ({100 * lo:+.2f} to {100 * hi:+.2f})",
                             "P": round(pv, 3)})

dnb_table = pd.DataFrame(dnb_rows)
dnb_table.to_csv(os.path.join(SAVE_DIR, "decision_curve_delta_nb.csv"), index=False)
print(dnb_table.to_string(index=False))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

UPD_SETS = [(5, (1, 18), "1–17 y, ≤5 min"), (5, (1, 15), "1–14 y, ≤5 min"), (30, (1, 15), "1–14 y, ≤30 min"), (60, (1, 15), "1–14 y, ≤60 min"),
            (30, (1, 18), "1–17 y, ≤30 min"), (60, (1, 18), "1–17 y, ≤60 min")]
UPD_K = 3.3
UPD_FOLDS = 5
UPD_REPEATS = 20
UPD_BOOT = 1000

UPD_STRATEGIES = [
    ("Original (no update)", "none", "fixed"),
    ("Threshold only (local)", "none", "local"),
    ("Intercept recalibration", "intercept", "fixed"),
    ("Logistic recalibration", "logistic", "fixed"),
    ("Logistic recalibration + local threshold", "logistic", "local"),
]
UPD_MODELS = {"Deep learning": ("dl", MAIN_THRESHOLD), "LR: ECG features": ("lr", lr_th)}


def expit(x):
    return 1 / (1 + np.exp(-x))


def update_probs(y_tr, p_tr, p_te, method):
    if method == "none":
        return p_te, p_tr
    if method == "intercept":
        a = calibration_in_the_large(y_tr, p_tr)
        return expit(a + logit(p_te)), expit(a + logit(p_tr))
    if method == "logistic":
        a, b = fit_platt(y_tr, p_tr)
        return apply_platt(p_te, a, b), apply_platt(p_tr, a, b)
    raise ValueError(method)


def cv_update(y, p, ids, method, threshold_mode, base_threshold, seed):
    p_out = np.zeros(len(y))
    pred_out = np.zeros(len(y), dtype=bool)
    thresholds = []
    cv = StratifiedGroupKFold(n_splits=UPD_FOLDS, shuffle=True, random_state=seed)
    for tr, te in cv.split(p, y, groups=ids):
        if 0 < y[tr].sum() < len(tr):
            p_te, p_tr = update_probs(y[tr], p[tr], p[te], method)
        else:
            p_te, p_tr = p[te], p[tr]
        th = select_threshold(y[tr], p_tr) if threshold_mode == "local" else base_threshold
        p_out[te] = p_te
        pred_out[te] = p_te >= th
        thresholds.append(th)
    return p_out, pred_out, thresholds


def upd_row(y, p, pred):
    tp, fn = int((pred & (y == 1)).sum()), int((~pred & (y == 1)).sum())
    tn, fp = int((~pred & (y == 0)).sum()), int((pred & (y == 0)).sum())
    brier = brier_score_loss(y, p)
    ref = y.mean() * (1 - y.mean())
    return {
        "AUROC": round(roc_auc_score(y, p), 3),
        "AUPRC": round(average_precision_score(y, p), 3),
        "Sensitivity": fmt(tp / (tp + fn), *exact_ci(tp, tp + fn)),
        "Specificity": fmt(tn / (tn + fp), *exact_ci(tn, tn + fp)),
        "PPV": fmt(tp / (tp + fp), *exact_ci(tp, tp + fp)) if tp + fp else "—",
        "Flagged (%)": round(100 * pred.mean(), 1),
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
        "Calib. intercept": round(calibration_in_the_large(y, p), 3),
        "Calib. slope": round(calibration_slope(y, p), 3),
        "O/E": round(y.sum() / p.sum(), 3),
        "Scaled Brier": round(1 - brier / ref, 3),
    }


upd_rows, upd_stab_rows, upd_cmp_rows, upd_store = [], [], [], {}
for window, (a0, a1), set_label in UPD_SETS:
    m = (dt_all <= window) & (age_all >= a0) & (age_all < a1)
    y = (k_all[m] <= UPD_K).astype(int)
    ids = np.asarray(ids_all)[m]
    probs = {"dl": p_all[m], "lr": lr_p_all[m]}
    print(f"{set_label}: {m.sum()} ECGs, {len(np.unique(ids))} patients, {y.sum()} hypokalemia")
    if y.sum() < UPD_FOLDS:
        print(f"  Only {y.sum()} cases: at most 1 per fold, updated parameters are unstable")

    preds_main = {}
    for model_name, (key, base_th) in UPD_MODELS.items():
        for strat_name, method, th_mode in UPD_STRATEGIES:
            p_upd, pred, ths = cv_update(y, probs[key], ids, method, th_mode, base_th, SEED)
            row = {"set": set_label, "n_ECG": int(m.sum()), "n_hypo": int(y.sum()), "model": model_name,
                   "strategy": strat_name,
                   "threshold": (f"{base_th:.4f}" if th_mode == "fixed"
                                 else f"{np.median(ths):.4f} ({min(ths):.4f}–{max(ths):.4f})"),
                   **upd_row(y, p_upd, pred)}
            upd_rows.append(row)
            upd_store[(set_label, model_name, strat_name)] = (y, p_upd, pred, ids)
            if strat_name == "Logistic recalibration + local threshold":
                preds_main[model_name] = (p_upd, pred)
                sens, spec, flag = [], [], []
                for r in range(UPD_REPEATS):
                    _, pr, _ = cv_update(y, probs[key], ids, method, th_mode, base_th, SEED + 1000 + r)
                    sens.append(pr[y == 1].mean()); spec.append((~pr[y == 0]).mean()); flag.append(100 * pr.mean())
                upd_stab_rows.append({"set": set_label, "model": model_name, "repeats": UPD_REPEATS,
                                      "Sensitivity median (min–max)": f"{np.median(sens):.3f} ({min(sens):.3f}–{max(sens):.3f})",
                                      "Specificity median (min–max)": f"{np.median(spec):.3f} ({min(spec):.3f}–{max(spec):.3f})",
                                      "Flagged % median (min–max)": f"{np.median(flag):.1f} ({min(flag):.1f}–{max(flag):.1f})"})

    (p_dl, pr_dl), (p_lr, pr_lr) = preds_main["Deep learning"], preds_main["LR: ECG features"]
    pos, neg = y == 1, y == 0
    b_sen, c_sen, pv_sen = mcnemar_exact(pr_dl[pos], pr_lr[pos])
    b_spe, c_spe, pv_spe = mcnemar_exact(~pr_dl[neg], ~pr_lr[neg])
    upd_cmp_rows.append({
        "set": set_label, "n_hypo": int(y.sum()),
        "ΔAUROC (DL − LR)": paired_delta(y, p_dl, p_lr, ids, roc_auc_score, n_boot=UPD_BOOT),
        "ΔAUPRC (DL − LR)": paired_delta(y, p_dl, p_lr, ids, average_precision_score, n_boot=UPD_BOOT),
        "Sens DL / LR": f"{pr_dl[pos].mean():.3f} / {pr_lr[pos].mean():.3f}",
        "Sens McNemar": f"DL only {b_sen} / LR only {c_sen}, P = {pv_sen:.3f}",
        "Spec DL / LR": f"{(~pr_dl[neg]).mean():.3f} / {(~pr_lr[neg]).mean():.3f}",
        "Spec McNemar": f"DL only {b_spe} / LR only {c_spe}, P = {pv_spe:.3f}",
        "Flagged DL / LR (%)": f"{100 * pr_dl.mean():.1f} / {100 * pr_lr.mean():.1f}",
    })

update_table = pd.DataFrame(upd_rows)
update_stability_table = pd.DataFrame(upd_stab_rows)
update_compare_table = pd.DataFrame(upd_cmp_rows)
update_table.to_csv(os.path.join(SAVE_DIR, "model_updating_external.csv"), index=False)
update_stability_table.to_csv(os.path.join(SAVE_DIR, "model_updating_stability.csv"), index=False)
update_compare_table.to_csv(os.path.join(SAVE_DIR, "model_updating_dl_vs_lr.csv"), index=False)

pd.set_option("display.width", 280)
for set_label in [s[2] for s in UPD_SETS]:
    sub = update_table[update_table["set"] == set_label]
    print(f"\n=== {set_label} (n_hypo = {sub['n_hypo'].iloc[0]}) ===")
    print(sub[["model", "strategy", "threshold", "AUROC", "Sensitivity", "Specificity", "Flagged (%)",
               "Calib. intercept", "Calib. slope", "O/E", "Scaled Brier"]].to_string(index=False))
print("\n=== Repeated fold assignments (logistic recalibration + local threshold) ===")
print(update_stability_table.to_string(index=False))
print("\n=== DL vs LR after updating both (logistic recalibration + local threshold) ===")
print(update_compare_table.to_string(index=False))

UPD_PLOT_SETS = ["1–14 y, ≤60 min", "1–17 y, ≤60 min"]
UPD_BINS = 8
UPD_STYLE = {("Deep learning", "Original (no update)"): ("#c0392b", "--"),
             ("Deep learning", "Logistic recalibration"): ("#c0392b", "-"),
             ("LR: ECG features", "Original (no update)"): ("#2c7fb8", "--"),
             ("LR: ECG features", "Logistic recalibration"): ("#2c7fb8", "-")}


def reliability(y, p, n_bins=UPD_BINS):
    edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, p, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        sel = idx == b
        if sel.any():
            xs.append(p[sel].mean()); ys.append(y[sel].mean())
    return np.asarray(xs), np.asarray(ys)


fig, axes = plt.subplots(1, len(UPD_PLOT_SETS), figsize=(6.2 * len(UPD_PLOT_SETS), 5.4), squeeze=False)
for ax, set_label in zip(axes[0], UPD_PLOT_SETS):
    top = 0
    for (model_name, strat_name), (color, ls) in UPD_STYLE.items():
        y, p_upd, _, _ = upd_store[(set_label, model_name, strat_name)]
        xs, ys = reliability(y, p_upd)
        top = max(top, xs.max(), ys.max())
        tag = "original" if strat_name.startswith("Original") else "recalibrated (CV)"
        ax.plot(xs, ys, color=color, ls=ls, marker="o", ms=4, lw=1.5, label=f"{model_name}, {tag}")
    top *= 1.1
    ax.plot([0, top], [0, top], color="#999", lw=0.8, ls=":")
    ax.set_xlim(0, top); ax.set_ylim(0, top)
    ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed proportion (K ≤ 3.3)")
    n_hypo = int(upd_store[(set_label, "Deep learning", "Original (no update)")][0].sum())
    ax.set_title(f"External {set_label} (hypokalemia = {n_hypo})", fontsize=10)
    ax.grid(alpha=0.2); ax.legend(fontsize=8)
fig.text(0.5, -0.02, f"Recalibration fitted in {UPD_FOLDS}-fold patient-level cross-validation within the external cohort; "
         "each ECG is evaluated with parameters fitted on other patients. Points = probability quantile bins.",
         ha="center", fontsize=9, color="#555")
fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "model_updating_calibration.png"), dpi=300, bbox_inches="tight")
fig.savefig(os.path.join(SAVE_DIR, "model_updating_calibration.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
from scipy.stats import fisher_exact, mannwhitneyu

STATEMENT_WINDOWS = [EXTERNAL_MAX_DT_MIN, 60]
STATEMENT_AGE_RANGE = tuple(EXTERNAL_AGE_RANGE)
FLAG_KEYS = [k for k in korea.keys() if str(k).startswith("flag_") and k != "flag_tall_t"]
print("External statement flags:", FLAG_KEYS)
flags_all = {k: np.asarray(korea[k]).astype(bool)[rows_all] for k in FLAG_KEYS}
if "flag_any_statement" not in flags_all:
    flags_all["flag_any_statement"] = np.any(np.column_stack([flags_all[k] for k in FLAG_KEYS]), axis=1)
FLAG_LIST = [k for k in FLAG_KEYS if k != "flag_any_statement"] + ["flag_any_statement"]


def rate(x):
    return f"{x.sum()}/{len(x)} ({100 * x.mean():.1f}%)" if len(x) else "—"


def fisher_p(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan
    return fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])[1]


fp_rows, fn_rows, reader_rows = [], [], []
for window in STATEMENT_WINDOWS:
    m = (dt_all <= window) & (age_all >= STATEMENT_AGE_RANGE[0]) & (age_all < STATEMENT_AGE_RANGE[1])
    y = (k_all[m] <= 3.3).astype(int)
    p = p_all[m]
    pred = p >= MAIN_THRESHOLD
    label = f"≤{window} min (n={m.sum()}, hypo={y.sum()})"
    neg, pos = y == 0, y == 1
    for f in FLAG_LIST:
        fl = flags_all[f][m]
        fp_rows.append({"window": label, "flag": f,
                        "FP (model +, K normal)": rate(fl[neg & pred]),
                        "TN (model −, K normal)": rate(fl[neg & ~pred]),
                        "Fisher P": round(fisher_p(fl[neg & pred], fl[neg & ~pred]), 4),
                        "median prob, flag+ vs flag− (K normal)":
                            f"{np.median(p[neg & fl]):.4f} vs {np.median(p[neg & ~fl]):.4f}" if (neg & fl).any() else "—",
                        "Mann–Whitney P": round(mannwhitneyu(p[neg & fl], p[neg & ~fl])[1], 4)
                        if (neg & fl).any() and (neg & ~fl).any() else np.nan})
        fn_rows.append({"window": label, "flag": f,
                        "TP (detected)": rate(fl[pos & pred]),
                        "FN (missed)": rate(fl[pos & ~pred]),
                        "Fisher P": round(fisher_p(fl[pos & pred], fl[pos & ~pred]), 4)})
        if fl.sum() > 0:
            tp, fn = int((fl & pos).sum()), int((~fl & pos).sum())
            tn, fpn = int((~fl & neg).sum()), int((fl & neg).sum())
            reader_rows.append({"window": label, "predictor": f"statement: {f}",
                                "Sensitivity": fmt(tp / max(tp + fn, 1), *exact_ci(tp, tp + fn)),
                                "Specificity": fmt(tn / max(tn + fpn, 1), *exact_ci(tn, tn + fpn)),
                                "Flagged (%)": round(100 * fl.mean(), 1)})
    tp, fn = int((pred & pos).sum()), int((~pred & pos).sum())
    tn, fpn = int((~pred & neg).sum()), int((pred & neg).sum())
    reader_rows.append({"window": label, "predictor": "Deep learning (threshold)",
                        "Sensitivity": fmt(tp / (tp + fn), *exact_ci(tp, tp + fn)),
                        "Specificity": fmt(tn / (tn + fpn), *exact_ci(tn, tn + fpn)),
                        "Flagged (%)": round(100 * pred.mean(), 1)})

pd.set_option("display.width", 250)
stmt_fp, stmt_fn, stmt_vs_model = pd.DataFrame(fp_rows), pd.DataFrame(fn_rows), pd.DataFrame(reader_rows)
for name, tab, title in [("statements_false_positive", stmt_fp, "Normokalemic ECGs: statements in false positives vs true negatives"),
                         ("statements_false_negative", stmt_fn, "Hypokalemia ECGs: statements in true positives vs false negatives"),
                         ("statements_vs_model", stmt_vs_model, "Statements vs model: hypokalemia detection")]:
    tab.to_csv(os.path.join(SAVE_DIR, f"{name}.csv"), index=False)
    print(f"\n=== {title} ===")
    print(tab.to_string(index=False))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

MEASURES_PATH = os.path.join("korea_measures.parquet")
DEV_CV_REPEATS = 20
DEV_BOOT = 1000

if not os.path.exists(MEASURES_PATH):
    raise FileNotFoundError(f"{MEASURES_PATH} not found")
meas = pd.read_parquet(MEASURES_PATH)
row_pos = np.asarray(korea["ecg_ids"]).astype(int)
if row_pos.max() >= len(meas):
    raise ValueError(f"Maximum ecg_id {row_pos.max()} exceeds the number of measurement rows {len(meas)}")

_chk = pd.DataFrame({"a": np.asarray(korea["ids"]).astype(str),
                     "b": meas["PatientID"].astype(str).to_numpy()[row_pos]})
_bad = max(_chk.groupby("a")["b"].nunique().max(), _chk.groupby("b")["a"].nunique().max())
if _bad != 1:
    raise ValueError("Patient IDs in the npz file and the measurement file do not match")
print(f"Rows matched: {len(row_pos)} ECGs, {_chk['a'].nunique()} patients")
del _chk


def _num(col):
    if col not in meas.columns:
        return np.full(len(meas), np.nan)
    return pd.to_numeric(meas[col], errors="coerce").to_numpy(dtype=float)


def _pick(*cols):
    out = np.full(len(meas), np.nan)
    for c in cols:
        v = _num(c)
        out = np.where(np.isnan(out), v, out)
    return out


DEVICE_COLS = {
    "heart rate": (("HeartRate", "Heartrate"), (30, 300)),
    "PR interval": (("PRInterval", "Meanprint"), (40, 400)),
    "QRS duration": (("QRSDuration", "Meanqrsdur"), (30, 250)),
    "QT interval": (("QTInterval", "Meanqtint"), (150, 700)),
    "QTc": (("QTCorrected", "Meanqtc", "QTCB"), (200, 750)),
    "P axis": (("PAxis", "Pfrontaxis"), (-180, 360)),
    "QRS axis": (("RAxis", "Qrsfrontaxis"), (-180, 360)),
    "T axis": (("TAxis", "Tfrontaxis"), (-180, 360)),
}
for c in ("PRInterval@Unit", "QRSDuration@Unit", "QTInterval@Unit", "QTCorrected@Unit"):
    if c in meas.columns:
        print(f"{c}: {meas[c].astype(str).value_counts().head(3).to_dict()}")

dev = {}
for name, (cols, (lo, hi)) in DEVICE_COLS.items():
    v = _pick(*cols)[row_pos]
    if "interval" in name or name in ("QRS duration", "QTc"):
        if np.nanmedian(v) < 5:
            v = v * 1000
    v[(v < lo) | (v > hi)] = np.nan
    dev[name] = v
print("Missing measurements:", {k: f"{100 * np.isnan(v).mean():.1f}%" for k, v in dev.items()})


_m60 = (dt_all <= 60) & (age_all >= EXTERNAL_AGE_RANGE[0]) & (age_all < EXTERNAL_AGE_RANGE[1])
DEV_SETS = {
    EXT_LABEL: (np.asarray(MAIN_PRED["external"]["rows"]), MAIN_PRED["external"]["y"],
                MAIN_PRED["external"]["prob"], MAIN_PRED["external"]["ids"]),
    f"External {AGE_TXT}, ≤60 min": (rows_all[_m60], (k_all[_m60] <= 3.3).astype(int), p_all[_m60], ids_all[_m60]),
}
sex_all_k = np.where(korea["_meta"][:, 0] < 0, np.nan, korea["_meta"][:, 0]).astype(float)
age_all_k = np.asarray(korea["age_years"], dtype=float)

single_rows = []
for set_label, (rows, y, p_dl, ids) in DEV_SETS.items():
    for name, v_all in dev.items():
        v = v_all[rows]
        ok = ~np.isnan(v)
        yy = y[ok]
        row = {"evaluation set": set_label, "measure": name, "n_ECG": int(ok.sum()), "n_hypo": int(yy.sum())}
        if 2 <= yy.sum() < len(yy):
            d, lo_, hi_, pv = paired_delta_auc(yy, p_dl[ok], v[ok], ids[ok], n_boot=DEV_BOOT)
            row.update({"AUROC (higher = hypo)": round(roc_auc_score(yy, v[ok]), 3),
                        "DL AUROC, same ECGs": round(roc_auc_score(yy, p_dl[ok]), 3),
                        "ΔAUROC (DL − measure)": f"{d:+.3f} ({lo_:+.3f} to {hi_:+.3f}), P = {pv:.3f}",
                        "median hypo / non-hypo": f"{np.median(v[ok][yy == 1]):.0f} / {np.median(v[ok][yy == 0]):.0f}"})
        else:
            row["note"] = "fewer than 2 cases"
        single_rows.append(row)
device_single_table = pd.DataFrame(single_rows)
print("\n=== Single device measurements: AUROC (higher value = hypokalemia; < 0.5 means opposite direction) ===")
print(device_single_table.to_string(index=False))

DEV_LR_SPECS = {
    "LR: device intervals (HR, PR, QRS, QTc)": ["heart rate", "PR interval", "QRS duration", "QTc"],
    "LR: device intervals + axes": ["heart rate", "PR interval", "QRS duration", "QTc", "P axis", "QRS axis", "T axis"],
    "LR: device intervals + axes + age + sex": ["heart rate", "PR interval", "QRS duration", "QTc",
                                                "P axis", "QRS axis", "T axis", "age", "sex"],
}


def _device_X(rows, names):
    cols = []
    for n in names:
        cols.append(age_all_k[rows] if n == "age" else sex_all_k[rows] if n == "sex" else dev[n][rows])
    return np.column_stack(cols)


def _cv_oof(X, y, ids, seed):
    _, lab = patient_level_label(ids, np.where(y == 1, 2, 0))
    n_splits = int(min(5, lab.sum()))
    oof = np.full(len(y), np.nan)
    sgk = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, va in sgk.split(X, y, groups=ids):
        if y[tr].sum() == 0:
            continue
        m = make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler(),
                          LogisticRegression(max_iter=2000)).fit(X[tr], y[tr])
        oof[va] = m.predict_proba(X[va])[:, 1]
    return oof


lr_dev_rows, device_lr_pred = [], {}
for set_label, (rows, y, p_dl, ids) in DEV_SETS.items():
    for label, names in DEV_LR_SPECS.items():
        X = _device_X(rows, names)
        oofs = np.array([_cv_oof(X, y, ids, SEED + r) for r in range(DEV_CV_REPEATS)])
        p_lr = np.nanmean(oofs, axis=0)
        ok = ~np.isnan(p_lr)
        per_rep = [roc_auc_score(y[~np.isnan(o)], o[~np.isnan(o)]) for o in oofs]
        d, lo_, hi_, pv = paired_delta_auc(y[ok], p_dl[ok], p_lr[ok], ids[ok], n_boot=DEV_BOOT)
        device_lr_pred[(set_label, label)] = p_lr
        lr_dev_rows.append({
            "evaluation set": set_label, "model": label, "n_ECG": int(ok.sum()), "n_hypo": int(y[ok].sum()),
            "LR AUROC (averaged OOF)": round(roc_auc_score(y[ok], p_lr[ok]), 3),
            "LR AUROC per repeat, median (range)": f"{np.median(per_rep):.3f} ({min(per_rep):.3f}–{max(per_rep):.3f})",
            "LR AUPRC": round(average_precision_score(y[ok], p_lr[ok]), 3),
            "DL AUROC": round(roc_auc_score(y[ok], p_dl[ok]), 3),
            "DL AUPRC": round(average_precision_score(y[ok], p_dl[ok]), 3),
            "ΔAUROC (DL − LR)": f"{d:+.3f} ({lo_:+.3f} to {hi_:+.3f}), P = {pv:.3f}",
        })
device_lr_table = pd.DataFrame(lr_dev_rows)
print(f"\n=== LR on device measurements: patient-level 5-fold CV within the external cohort × {DEV_CV_REPEATS} ===")
print(device_lr_table.to_string(index=False))

dist_rows = []
for set_label, (rows, y, _, _) in DEV_SETS.items():
    src = meas.iloc[row_pos[rows]].reset_index(drop=True)
    for col in ("Department_df11", "Department@code", "HighPass", "LowPass", "AC", "SignalBandwidth"):
        if col not in src.columns:
            continue
        vals = src[col].astype(str).replace({"nan": "missing", "None": "missing", "": "missing"})
        for val, n in vals.value_counts().head(15).items():
            dist_rows.append({"evaluation set": set_label, "variable": col, "value": val, "n_ECG": int(n),
                              "%": round(100 * n / len(vals), 1),
                              "n_hypo": int(y[(vals == val).to_numpy()].sum())})
external_setting_table = pd.DataFrame(dist_rows)
print("\n=== Care setting and filter settings (top 15 values) ===")
print(external_setting_table.to_string(index=False))

device_single_table.to_csv(os.path.join(SAVE_DIR, "device_measures_single_auroc.csv"), index=False)
device_lr_table.to_csv(os.path.join(SAVE_DIR, "device_measures_lr_cv.csv"), index=False)
external_setting_table.to_csv(os.path.join(SAVE_DIR, "external_care_setting_filters.csv"), index=False)
print("\nSaved: device_measures_single_auroc.csv, device_measures_lr_cv.csv, external_care_setting_filters.csv")